# 第52课 · Audio Core 完结 — 特征工程收口，38 个单元测试全绿，面试证据整理

**目标**：跑通 `tests/audio/` 全部测试，分析性能，完成 Month 1 通关标志。

🔗 Aurora 连接：
- `src/aurora/audio/io.py` — WAV 读写
- `src/aurora/audio/windows.py` — Hann / Hamming
- `src/aurora/audio/transforms.py` — DFT / FFT / IFFT / RFFT
- `src/aurora/audio/stft.py` — 分帧（framing） + 短时傅里叶变换
- `src/aurora/audio/mel.py` — Hz↔Mel 转换、mel 滤波器组、mel 频谱
- `src/aurora/audio/mfcc.py` — DCT-II、MFCC

← **上一课**　[L51 · MFCC 工程实战](L51_real_audio.ipynb)

> 上节课学习了 **MFCC 工程实战**：在真实 WAV 音频上提取特征，librosa 对答案。  
> 本课将探讨 **Audio Core 完结**。

## 本课剧情：怎么证明你"真的写了"FFT？

面试官说："你说你从零实现了 FFT 和 MFCC——给我看证据。"

这个问题有两层：
1. **代码层**：测试套件能覆盖边界情况、数值精度、形状正确性吗？
2. **理解层**：不看代码，能白板推导完整公式链吗？

本节是 Audio Core 的收口仪式——跑通 `tests/audio/` 全部 38 个测试，整理"面试证据"。

**从 WAV 到 MFCC 的完整公式链**（6 步记忆法）：

```
x[n]  → STFT: X[t,k] = Σ x[t·hop+n]·w[n]·e^{-2πikn/N}
     → power: P[t,k] = |X[t,k]|²
     → Mel: M[t,m] = Σ_k P[t,k]·H_m[k]        (三角滤波器)
     → log: L[t,m] = log(M[t,m] + ε)
     → DCT: C[t,c] = Σ_m L[t,m]·cos(π·c·(2m+1)/(2N))·w(c)
     → 截断: MFCC[t, :n_mfcc]
```

**测试套件的三种断言类型**：

| 断言类型 | 示例 | 意义 |
|---|---|---|
| Shape | `assert S.shape == (n_frames, n_bins)` | 维度正确 |
| Numerical | `np.allclose(my_fft, np.fft.fft, atol=1e-10)` | 数值精度 |
| Property | `assert X[N-k] == conj(X[k])` | 数学性质 |

本节目标：`make test` 零失败，整理 5 个能在面试中白板推导的关键公式。

In [ ]:
import subprocess
import numpy as np

## 1. 从信号到特征的完整管线

```
WAV → 分帧(frame_signal) → 加窗(hann/hamming) → FFT(rfft)
    → |·|² → mel 滤波(mel_filterbank) → log → DCT-II → MFCC
```

每一步的数学形式：

- **分帧**：把长为 `T` 的信号切成长 `N`、步长 `H` 的帧。不填充时共 `1 + (T-N)//H` 帧；aurora 默认 `center=True`（先在两端各反射填充 `N//2`），帧数变为 `1 + T//H`。
- **加窗**：逐元素乘以 `w[n] = 0.5 * (1 - cos(2π n / N))`（Hann 周期窗）。
- **RFFT**：`X[k] = sum_{n=0}^{N-1} x[n] * e^{-j 2π k n / N}`，保留 `N//2+1` 个非冗余频点。
- **Mel 滤波**：`M[m] = sum_k fb[m,k] * |X[k]|²`，三角滤波器组（triangular filter bank），`fb` 形状 `(n_mels, N//2+1)`。
- **对数压缩（log compression）**：`log_mel = 10 * log10(M + eps)`，模拟人耳响度感知的对数特性。
- **DCT-II（正交归一化）**：`C[i] = scale[i] * sum_m log_mel[m] * cos(π/M * (m+0.5) * i)`，前 13 个系数即 MFCC。

In [ ]:
from aurora.audio import sine, stft, mel_spectrogram, mfcc

sr = 16000
x = sine(440.0, 1.0, sr)          # 1 秒 440 Hz 纯音

S = stft(x, n_fft=1024, hop_length=256)          # (n_frames, 513) complex
M = mel_spectrogram(x, sr, n_fft=1024, hop_length=256, n_mels=80)  # (n_frames, 80)
C = mfcc(x, sr, n_mfcc=13, n_fft=1024, hop_length=256)             # (n_frames, 13)

print(f"STFT  shape: {S.shape}  dtype: {S.dtype}")
print(f"Mel   shape: {M.shape}  dtype: {M.dtype}")
print(f"MFCC  shape: {C.shape}  dtype: {C.dtype}")
print(f"MFCC[0, :5] = {C[0, :5].round(4)}")
assert np.iscomplexobj(S)
assert M.shape[1] == 80
assert C.shape[1] == 13
print("✅ 管线形状全部正确")

## 2. 测试套件验证

`tests/audio/` 下有两个测试文件：

| 文件 | 覆盖范围 | 关键断言 |
|------|----------|----------|
| `test_transforms.py` | DFT / FFT / IFFT / RFFT | `atol=1e-9` 对比 `numpy.fft` |
| `test_features.py` | windows / STFT / mel / WAV I/O | `atol=1e-12` 窗函数 / DCT-II |
| `test_features.py` (MFCC) | MFCC shape only | 仅形状断言 `C.shape[1] == 13`，**暂无数值 atol 测试** |

关键测试点举例：

```python
# FFT 和 NumPy 逐元素一致
np.testing.assert_allclose(T.fft(x), np.fft.fft(x), atol=1e-9)

# DCT-II 手动公式
basis = np.cos(np.pi / n * np.outer(k, k + 0.5))
ref = x @ basis.T * scale
np.testing.assert_allclose(dct_ii(x), ref, atol=1e-12)

# Hann 窗与 numpy.hanning 一致（对称模式）
np.testing.assert_allclose(hann(64, periodic=False), np.hanning(64), atol=1e-12)
```

这些测试的设计原则：每个函数都有一个独立于实现的参考答案（公式或 NumPy），只要数学正确就一定通过。

In [ ]:
import sys
from pathlib import Path

result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/audio/", "-v", "--tb=short", "--no-header"],
    capture_output=True, text=True,
    cwd=subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip(),  # aurora repo root
)
print(result.stdout[-4000:] if len(result.stdout) > 4000 else result.stdout)
if result.returncode != 0:
    print(result.stderr[-2000:])
assert result.returncode == 0, "❌ 有测试失败，请检查上方输出"
print("✅ tests/audio/ 全部通过")

## 3. Month 1 交付标准

通关的三层验证：

**层1 — 代码能跑**：`make test` 零失败，`make demo` 输出正确频谱图。

**层2 — 数学能说清**：能对着空白文件重写任意一个函数。逐行对应公式，不依赖记忆：
- `dct_ii`：`basis[k, m] = cos(π/N * (m+0.5) * k)`，正交归一化系数 `sqrt(2/N)`，首项 `sqrt(1/N)`。
- `mel_filterbank`：三角滤波器顶点坐标在 mel 域均匀分布，转换回 Hz 后不均匀。
- `fft`：Cooley-Tukey 蝶形分解，`N=2^k` 时将 `O(N²)` 降为 `O(N log N)`。

**层3 — 能对比差异**：能解释 aurora 和 librosa 输出微小不同的原因（归一化策略、mel 刻度参数、中心填充行为）。

```
检查清单
[ ] pytest tests/audio/  →  全绿
[ ] 能手写 dct_ii(x)，解释每行
[ ] 能手写 mel_filterbank(n_mels, n_fft, sr)，解释三角形顶点坐标
[ ] 能解释 hann(N, periodic=True) 和 periodic=False 的区别
[ ] 能说出 stft 输出形状 (n_frames, N//2+1) 的来由
```

In [ ]:
from aurora.audio import (
    sine, hann, hamming, dct_ii, mel_filterbank,
    frame_signal, stft, mel_spectrogram, mfcc,
    hz_to_mel, mel_to_hz, read_wav, write_wav
)

# 检查1：DCT-II 手动验证
x_test = np.array([1.0, 2.0, 3.0, 4.0])
n = len(x_test)
k = np.arange(n)
basis = np.cos(np.pi / n * np.outer(k, k + 0.5))
scale = np.full(n, np.sqrt(2.0 / n)); scale[0] = np.sqrt(1.0 / n)
ref_dct = x_test @ basis.T * scale
np.testing.assert_allclose(dct_ii(x_test), ref_dct, atol=1e-12)
print("✅ DCT-II 手动公式验证通过")

# 检查2：mel 滤波器组形状和非负性
fb = mel_filterbank(n_mels=40, n_fft=1024, sample_rate=16000)
assert fb.shape == (40, 513) and np.all(fb >= 0.0)
print(f"✅ mel_filterbank shape={fb.shape}, 全非负, 各行峰值 > 0: {np.all(fb.max(axis=1) > 0)}")

# 检查3：Hann 窗周期/对称
per = hann(8, periodic=True)
sym = hann(9, periodic=False)
np.testing.assert_allclose(per, sym[:-1], atol=1e-12)
print("✅ hann periodic=True 等于 symmetric N+1 去掉最后一点")

# 检查4：STFT 形状
x_1s = sine(440.0, 1.0, 16000)
S = stft(x_1s, n_fft=1024, hop_length=256)
assert S.shape[1] == 1024 // 2 + 1
print(f"✅ stft 输出 {S.shape}，频率维 = N//2+1 = {1024//2+1}")

# 检查5：mel 转换可逆
freqs = np.array([0.0, 100.0, 440.0, 1000.0, 8000.0])
np.testing.assert_allclose(mel_to_hz(hz_to_mel(freqs)), freqs, atol=1e-6)
print("✅ hz_to_mel → mel_to_hz 往返误差 < 1e-6 Hz")

## 参数实验：aurora.mfcc vs librosa.feature.mfcc

**参数**：`n_fft=1024, hop_length=256, n_mels=80, n_mfcc=13`，信号为 1 秒 440 Hz 纯音。

**预期现象**：
- 形状相同 `(n_frames, 13)`（两者默认都 `center=True`，帧数一致），数值相近但不完全一致。
- 主要差异来源：
  1. mel 刻度与归一化：aurora 用 HTK 公式 `2595 * log10(1 + f/700)` 且三角滤波器不归一化；librosa 默认 **Slaney** 刻度（`htk=False`）加 `norm="slaney"` 面积归一化——这是数值差异的主要来源。
  2. 填充模式：都在首尾各填充 `n_fft//2`，但 aurora 用反射填充，librosa 新版默认零填充，只影响首尾几帧。
  3. power 参数：librosa 默认对功率谱（`|X|²`）建 mel，aurora `mel_spectrogram` 同样是功率谱，这一点一致。

运行下方 code 格，观察 RMSE 数量级和第 0 帧差异。

In [ ]:
try:
    import librosa
    HAS_LIBROSA = True
except ImportError:
    HAS_LIBROSA = False
    print("librosa 未安装，跳过对比（pip install librosa 后重试）")

if HAS_LIBROSA:
    from aurora.audio import sine, mfcc as aurora_mfcc

    sr = 16000
    x = sine(440.0, 1.0, sr).astype(np.float32)

    # aurora（center=True，反射填充）
    C_aurora = aurora_mfcc(x.astype(np.float64), sr,
                           n_mfcc=13, n_fft=1024, hop_length=256, n_mels=80)

    # librosa（center=True，默认零填充）
    C_lib_center = librosa.feature.mfcc(
        y=x, sr=sr, n_mfcc=13, n_fft=1024, hop_length=256, n_mels=80
    ).T  # librosa 返回 (n_mfcc, n_frames), 转置为 (n_frames, n_mfcc)

    print(f"aurora  MFCC shape: {C_aurora.shape}")
    print(f"librosa MFCC shape: {C_lib_center.shape}")

    # 对齐帧数后比较
    n_common = min(C_aurora.shape[0], C_lib_center.shape[0])
    rmse = np.sqrt(np.mean((C_aurora[:n_common] - C_lib_center[:n_common])**2))
    print(f"前 {n_common} 帧 RMSE = {rmse:.4f}")
    print(f"aurora  第0帧前5系数: {C_aurora[0, :5].round(3)}")
    print(f"librosa 第0帧前5系数: {C_lib_center[0, :5].round(3)}")
    print()
    print("差异来源分析：")
    print(f"  帧数差 = {C_lib_center.shape[0] - C_aurora.shape[0]}（两者都 center=True，应为 0）")
    print("  数值差异主要来自 mel 刻度：aurora 用 HTK 且不归一化，librosa 默认 Slaney + norm='slaney'")

## 附录 · Audio Core 结业：你已具备的能力清单

- [ ] 能从空白重写 `fft` / `stft` / `mel_filterbank` / `mfcc`（口述 + 白板）
- [ ] `make test` audio 全绿
- [ ] 能解释 MFCC 五段 shape：`(T, n_mels)` → `(T, 13)`
- [ ] **下一步不是更多 DSP**，而是 **L54：学模型如何从新数据中学习**

> 这张清单不是额外作业，而是 Month 1 的收束：四项都能口述后，再切到深度学习入口就不会断层。

## 本课收束

`mfcc()`、`mel_filterbank()`、`dct_ii()`、`stft()` 构成 Aurora Audio Analysis Engine 的完整特征提取栈，所有函数已通过 `tests/audio/` 数值断言（`atol` 最严达 `1e-12`）验证与 NumPy / 公式参考一致。
Month 1 的交付物是一个**零黑盒的纯 NumPy 音频特征管线**，覆盖从 WAV I/O 到 13 维 MFCC 向量的全部步骤。
`aurora.audio` 模块已可作为后续深度学习模块（L54+）的特征提取后端，直接调用 `mfcc()` 输出 `(n_frames, 13)` 张量。
下一课（L53）将用图形化方式展示 MFCC 的完整提取流水线：波形 → 声谱图 → Mel 谱 → 倒谱系数，逐层可视化。

## ✏️ 白板挑战：Audio Core 面试六连击（目标 15 分钟）

盖上屏幕，纸上推导：

**问 1**：写出 DFT 公式，说明 FFT 如何把复杂度从 O(N²) 降到 O(N·logN)。  
（分治：X[k]=E[k]+W^k·O[k]；log₂N 层；每层 N/2 蝶形）

**问 2**：Hann 窗公式是什么？为什么它能减少频谱泄漏？  
（w[n]=0.5(1-cos(2πn/N))；两端渐变到0，消除边界跳变）

**问 3**：Mel 滤波器组的三步构造流程：  
① Mel域均匀取中心点 → ② 反变换到Hz → ③ 构造三角形 → 矩阵形状?

**问 4**：DCT-II 正交归一化的归一化系数 w(k) 是什么？  
（k=0: 1/√N，k>0: √(2/N)；满足 D·Dᵀ=I）

**问 5**：MFCC 完整流水线的 shape 流向（数字填空）：  
`sr=16000, n_fft=1024, hop=256, dur=1s, n_mels=40, n_mfcc=13`  
→ n_frames=?，各步输出 shape?

**问 6**：为什么测试中用 `atol=1e-8` 而不是 `atol=0`？  
（浮点累积误差；log₂N 层 FFT 误差∝N·ε_machine）

推导完成后运行下面格验证数值。

In [ ]:
# ✏️ 面试六连击对答案格
import numpy as np, sys
sys.path.insert(0, 'src')

# 问1：FFT 复杂度 — 蝶形验证
N = 8
n_layers = int(np.log2(N))
butterflies = N // 2 * n_layers
dft_ops = N * N
speedup = dft_ops / butterflies
assert n_layers == 3 and butterflies == 12 and speedup > 5
print(f"Q1 ✅  N={N}: DFT {dft_ops}次，FFT {butterflies}蝶形，加速 {speedup:.1f}×")
# N 越大优势越大：N=1024 时加速 ~205×
N_big = 1024
speedup_big = N_big**2 / (N_big // 2 * int(np.log2(N_big)))
assert speedup_big > 100
print(f"      N={N_big}: DFT {N_big**2}次 vs FFT {N_big//2*int(np.log2(N_big))}蝶形，加速 {speedup_big:.0f}×")

# 问2：Hann 窗边界 = 0
from aurora.audio.windows import hann
w = hann(16)
assert np.isclose(w[0], 0.0, atol=1e-12)
print(f"Q2 ✅  Hann窗 w[0]={w[0]:.1f}（边界=0，消除跳变）")

# 问3：Mel filterbank shape
from aurora.audio.mel import mel_filterbank
fb = mel_filterbank(n_mels=40, n_fft=1024, sample_rate=16000)
assert fb.shape == (40, 513) and np.all(fb >= 0)
print(f"Q3 ✅  mel_filterbank shape={fb.shape}，权重非负")

# 问4：DCT 正交性
from aurora.audio.mfcc import dct_ii
N_dct = 13
k_arr = np.arange(N_dct)
n_arr = np.arange(N_dct)
D = np.cos(np.pi * np.outer(k_arr, 2*n_arr+1) / (2*N_dct))
w0 = 1.0 / np.sqrt(N_dct)
wk = np.sqrt(2.0 / N_dct)
D[0] *= w0
D[1:] *= wk
err = np.max(np.abs(D @ D.T - np.eye(N_dct)))
assert err < 1e-12
print(f"Q4 ✅  DCT矩阵正交性：max|D·Dᵀ-I|={err:.2e}")

# 问5：shape 流向（aurora stft 默认 center=True：n_frames = 1 + N//hop）
sr, n_fft, hop, dur = 16000, 1024, 256, 1.0
N = int(sr * dur)
n_frames = 1 + N // hop
n_bins = n_fft // 2 + 1
n_mels, n_mfcc = 40, 13
assert n_frames == 63 and n_bins == 513
print(f"Q5 ✅  n_frames={n_frames}: power({n_frames},{n_bins})→Mel({n_frames},{n_mels})→MFCC({n_frames},{n_mfcc})")

# 问6：浮点误差 — FFT 误差量级
x = np.random.randn(1024)
err_fft = np.max(np.abs(np.fft.fft(x) - np.fft.fft(x.copy())))
# 验证 atol=1e-8 合理（浮点机器精度 × N·log₂N）
machine_eps = np.finfo(float).eps
n_fft2 = 1024
error_bound = machine_eps * n_fft2 * np.log2(n_fft2)
print(f"Q6 ✅  机器精度={machine_eps:.2e}，N·log₂N误差上界≈{error_bound:.2e}（1e-8 安全）")
print("\n🎉 Audio Core 面试六连击通过！你的实现有扎实数学基础。")

In [ ]:
# ✏️ 本课自评（Audio Core 完结）
l52_review = {
    "test_suite_green":        None,  # make test 全部 38 个测试通过？True/False
    "fft_butterfly_formula":   None,  # 白板能写出 X[k]=E[k]+W^k·O[k]？True/False
    "mel_dct_pipeline":        None,  # 能从 shape 流向手算 n_frames 和各步 shape？True/False
    "interview_ready":         None,  # 能不看代码推导完整 6 步公式链？True/False
    "whiteboard_passed":       None,  # 面试六连击白板全部完成？True/False
}

unfilled = [k for k, v in l52_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l52_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强（面试前必须打磨）：{weak}')
else:
    print('✅ L52 全部通关！Audio Core 完结——进入 L53：MFCC 图形化')

---

→ **下一课**　[L53 · MFCC 图形化](L53_visual_mfcc.ipynb)

> 下节课将学习 **MFCC 图形化**：波形 → 声谱图 → Mel 谱 → 倒谱系数，逐层图示。